# EvalPlus: carregar → gerar → validar

Demonstração do fluxo da Fase 1 com EvalPlus (HumanEval+ / MBPP+):

1. carregar um item do dataset (`load_evalplus`);
2. enviar o prompt para um modelo (Ollama) e obter o código gerado;
3. validar/classificar o código em `syntax | runtime | functional | correct`.

> Requisito: Ollama rodando (`ollama serve`) e o modelo escolhido instalado (`ollama pull <modelo>`).

## 1. Pegar um item do dataset

In [ ]:
from src.dataset.load import load_evalplus

humaneval = load_evalplus("humaneval")
mbpp = load_evalplus("mbpp")

print("HumanEval+:", len(humaneval), "tarefas")
print("MBPP+:", len(mbpp), "tarefas")

example = [e for e in humaneval if e.task_id == "HumanEval/0"][0]

print("\n--- Exemplo ---")
print("task_id:", example.task_id)
print("benchmark:", example.benchmark_name)
print("entry_point:", example.entry_point
print("is_completion:", example.is_completion)
print("function_signature:", example.function_signature)
print("\nprompt:\n", example.prompt)
print("\nbase_input (3 primeiros):", example.base_input[:3])
print("plus_input (total):", len(example.plus_input))

## 2. Mandar para o modelo

`generate_code` já monta o prompt certo conforme o benchmark:

- **humaneval** (`is_completion=True`): pede para completar o corpo da função;
- **mbpp** (`is_completion=False`): pede a função completa.

In [ ]:
from src.models.ollama_handler import OllamaHandler

model = "gemma3:1b"  # troque por qualquer modelo em config.yaml

handler = OllamaHandler(model)
code = handler.generate_code(example, temperature=0.0)
handler.close()

print(code)

## 3. Validar / classificar

`run_evalplus_tests` executa o código contra os testes **Base** e **Extra** do EvalPlus e devolve:

- `level_name`: `syntax | runtime | functional | correct`;
- `tc_ok` / `tc_fail`: **quantos testes passaram / falharam**;
- `evidences`: lista de justificativas (mensagem do erro ou casos de teste que falharam, com entrada/saída).

In [ ]:
from src.evaluations.classify import run_evalplus_tests

result = run_evalplus_tests(code, example, timeout_seconds=30)

print("level:", result.level_name)
print("tc_ok:", result.tc_ok, "| tc_fail:", result.tc_fail)
print("exception_type:", result.exception_type)
print("failure_stage:", result.failure_stage)
print("\nevidências:")
for ev in result.evidences[:10]:
    print(" -", ev)
if len(result.evidences) > 10:
    print(f" ... e mais {len(result.evidences) - 10} evidências")

## 4. Como funciona a validação?

`run_evalplus_tests` roda o código gerado num **subprocesso isolado** e compara com a solução de referência (o `canonical_solution` do EvalPlus).

### Passo a passo

1. **`compile()`** o código no processo pai → se der `SyntaxError`, vira `syntax` (nem precisa executar).
2. **Executa** o código no subprocesso (harness) e obtém a função `entry_point`.
3. Para **cada teste** de `base_input` e depois de `plus_input`:
   - chama a **solução canônica** para obter a saída esperada (`expected`);
   - chama a **função candidata** e compara com o esperado;
   - conta `tc_ok`/`tc_fail` e registra as `evidences`.
4. **Captura de erros**: cada chamada fica em `try/except`. Se lançar exceção (`NameError`, `TypeError`, …) ou timeout → `runtime`. Se executar mas a saída diferir → `functional`. Se passar tudo (Base **e** Extra) → `correct`.

### Precedência

```
não compila        -> syntax
lança exceção      -> runtime
saída errada       -> functional
passa Base + Extra -> correct
```

## 5. Exemplo com MBPP+ (função completa)

No MBPP+ o modelo deve devolver a função inteira (com `def`).

In [ ]:
example_mbpp = [e for e in mbpp if e.task_id == "Mbpp/11"][0]

print("task_id:", example_mbpp.task_id)
print("is_completion:", example_mbpp.is_completion)
print("function_signature:", example_mbpp.function_signature)
print("\nprompt:\n", example_mbpp.prompt)

handler = OllamaHandler(model)
code_mbpp = handler.generate_code(example_mbpp, temperature=0.0)
handler.close()

print("\nCódigo gerado:\n", code_mbpp)

result_mbpp = run_evalplus_tests(code_mbpp, example_mbpp, timeout_seconds=30)
print("\nlevel:", result_mbpp.level_name, "| tc_ok:", result_mbpp.tc_ok, "| tc_fail:", result_mbpp.tc_fail)

## 6. Testar as 4 classes sem Ollama (sintético)

Para conferir o classificador sem depender de modelo:

In [ ]:
body_correto = example.canonical_solution[len(example.prompt):]  # corpo canônico

casos = {
    "correct": body_correto,
    "functional": "\n    return False\n",
    "runtime": "\n    return undefined_var\n",
    "syntax": "\n    return (1 +\n",
}

for nome, codigo in casos.items():
    r = run_evalplus_tests(codigo, example, timeout_seconds=30)
    print(f"{nome:12s} -> {r.level_name:12s} tc_ok={r.tc_ok} tc_fail={r.tc_fail} | {r.evidences[:1]}")